In [24]:
import pandas as pd
# import scanpy as sc
import numpy as np
import anndata as ad
import json

from choose_protein_coding import list_of_protein_coding_genes

In [25]:
# configure
raw_data_dir = "../data/raw_tsv_data"
filename = "TCGA-LUAD.star_tpm"

data_for_mlp_dir = "../data/0_data_for_mlp"
data_for_scgpt_dir = "../data/0_adata_for_scgpt"

gene_info_file = "../data/gene_info.csv"

# for MLP

In [26]:
df_original = pd.read_csv(f'{raw_data_dir}/{filename}.tsv', delimiter='\t', index_col=False)

In [27]:
df = df_original.rename(columns={'Ensembl_ID': 'Unnamed: 0-1'})
df.head()

,Unnamed: 0-1,TCGA-38-7271-01A,TCGA-55-7914-01A,TCGA-95-7043-01A,TCGA-73-4658-01A,TCGA-86-8076-01A,TCGA-55-7726-01A,TCGA-44-6147-01A,TCGA-50-5932-01A,TCGA-44-2661-01A,...,TCGA-50-5946-02A,TCGA-86-7713-01A,TCGA-86-8073-01A,TCGA-44-2662-01B,TCGA-MN-A4N4-01A,TCGA-53-7626-01A,TCGA-62-A46O-01A,TCGA-44-A47G-01A,TCGA-55-6969-01A,TCGA-55-6969-11A
0,ENSG00000000003.15,4.993860,5.572080,5.037615,6.157945,5.061914,5.071063,5.385517,6.225234,5.659973,...,5.877877,6.605066,5.321856,5.100397,5.620695,5.077734,6.845089,5.103242,4.201736,3.721241
1,ENSG00000000005.6,0.000000,0.000000,0.000000,3.598949,0.000000,0.000000,0.196985,0.000000,0.000000,...,0.000000,0.000000,0.000000,1.341758,0.000000,0.000000,0.000000,0.000000,0.000000,0.101650
2,ENSG00000000419.13,5.876966,6.447268,6.706663,6.168664,5.867071,7.343107,6.391027,6.585406,6.432363,...,6.874345,6.771603,6.987923,6.713725,6.928426,6.241032,6.418473,5.584301,6.376573,6.094359
3,ENSG00000000457.14,2.954848,3.269826,2.656977,2.761775,3.112600,2.552623,3.426198,3.774302,3.060912,...,3.099379,4.229765,3.100843,4.618344,3.059217,3.313318,3.158644,2.693632,2.542852,2.696061
4,ENSG00000000460.17,1.858379,1.965212,1.934139,1.726788,1.763412,2.477237,2.423430,2.314232,1.848958,...,3.202151,4.033529,2.616358,4.613820,2.562841,2.095047,3.221460,1.687509,2.272292,1.212756


In [28]:
# choose 01A for samples with duplicates

col_df = pd.DataFrame({"col": df.columns})

# prefix bez ostatniej litery
col_df["prefix"] = col_df["col"].str[:-1]

# suffix = ostatnia litera
col_df["suffix"] = col_df["col"].str[-1]

col_df["is_A"] = (col_df["suffix"] == "A").astype(int)
col_df = col_df.sort_values(
    ["prefix", "is_A"],
    ascending=[True, False]
)
selected_cols = col_df.drop_duplicates("prefix")["col"].tolist()

if 'Unnamed: 0-1' in selected_cols:
    selected_cols.remove('Unnamed: 0-1')
selected_cols.insert(0, 'Unnamed: 0-1')


In [30]:
df = df[selected_cols]
df.columns = df.columns.str.split("-").str[:-1].str.join("-")

df = df.T
df.columns = df.iloc[0]   # pierwszy wiersz → nazwy kolumn
df = df.iloc[1:]

In [31]:
df.head()

Unnamed: 0,ENSG00000000003.15,ENSG00000000005.6,ENSG00000000419.13,ENSG00000000457.14,ENSG00000000460.17,ENSG00000000938.13,ENSG00000000971.16,ENSG00000001036.14,ENSG00000001084.13,ENSG00000001167.14,...,ENSG00000288661.1,ENSG00000288662.1,ENSG00000288663.1,ENSG00000288665.1,ENSG00000288667.1,ENSG00000288669.1,ENSG00000288670.1,ENSG00000288671.1,ENSG00000288674.1,ENSG00000288675.1
TCGA-05-4244,5.989748,0.0,6.113669,3.558366,3.087055,5.084413,4.466438,5.89336,4.207182,5.412507,...,0.0,0.0,0.398898,0.0,0.0,0.0,4.535829,0.0,0.198368,0.654527
TCGA-05-4249,5.799851,0.0,6.571369,3.850479,2.478014,4.408732,4.354375,5.589443,3.270903,6.558744,...,0.0,0.0,0.543001,0.0,0.947255,0.0,3.542667,0.0,0.145221,0.653427
TCGA-05-4250,6.079143,0.253989,7.096947,2.634686,2.966523,4.682023,5.051864,6.079459,4.075618,5.649613,...,0.0,0.0,0.188528,0.0,0.0,0.020484,1.72076,0.0,0.016924,0.306612
TCGA-05-4382,4.79763,0.0,6.212949,2.499502,2.26138,5.460372,5.1258,6.028671,4.266592,4.955322,...,0.0,0.0,0.422664,0.0,0.0,0.011209,3.347255,0.0,0.050745,0.605494
TCGA-05-4384,5.064896,0.0,6.382247,3.819045,2.160952,5.127361,4.453142,5.448676,6.173749,5.096372,...,0.0,0.0,0.603692,0.0,0.0,0.0,3.949105,0.0,0.125916,0.606253


In [32]:
df.columns = df.columns.str.split(".").str[0]

features = pd.read_csv(gene_info_file)
features = features[["feature_id", "feature_name"]]

id_to_symbol = dict(
    zip(features["feature_id"], features["feature_name"])
)

df = df.rename(columns=id_to_symbol)

df = df.loc[:, df.columns.notnull()]
df = df.loc[:, ~df.columns.duplicated()]

df.shape

(577, 60616)

In [34]:
# keep only protein-coding genes

gene_list = df.columns
gene_list = gene_list[1:]
protein_coding_gene_list = list_of_protein_coding_genes(gene_list)

df_filtered = df[ df.columns.intersection(protein_coding_gene_list) ]


In [35]:
df_filtered = df_filtered[~df_filtered.index.duplicated(keep="first")]


In [36]:
# save to file
df_filtered.to_csv(f"{data_for_mlp_dir}/{filename}.csv")


# for scGPT

In [37]:
# create anndata df

X = df_filtered.values.astype(np.float32)

obs = pd.DataFrame(index=df_filtered.index)
obs["sample"] = df_filtered.index

var = pd.DataFrame(index=df_filtered.columns)
var["gene_name"] = df_filtered.columns

adata = ad.AnnData(
    X=X,
    obs=obs,
    var=var
)

adata

AnnData object with n_obs × n_vars = 517 × 20260
    obs: 'sample'
    var: 'gene_name'

In [38]:
adata.write(f"{data_for_scgpt_dir}/adata_{filename}.h5ad")
